# Imports & Load Model

In [1]:
from transformers import pipeline
import pandas as pd

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

d:\customer-review-summarization\env\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

d:\customer-review-summarization\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SourceCode\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Load Test Data

In [2]:
df = pd.read_csv("../data/processed/test.csv")
df.head()


,product_name,product_price,Rate,Review,Summary,Sentiment,sentiment_label
0,Flipkart SmartBuy CFXB18 Electric Rice Cooker?...,1599,5,super!,good,positive,2
1,Dettol Effective Protection Antiseptic LiquidÐ...,349,5,must buy!,very good product,positive,2
2,DOMS Pencil Smart Kit,449,5,worth every penny,its very good and all are bright colors,positive,2
3,Fun and Flex Solid Jungle Theme Animal Face Fo...,268,2,expected a better product,very bad all baloon r based,negative,0
4,HP 680 Black Ink Cartridge,852,5,great product,fantastic,positive,2


# Generate Summaries (Small Batch)

In [3]:
sample_reviews = df['Review'].sample(5).tolist()

summaries = summarizer(
    sample_reviews,
    max_length=60,
    min_length=25,
    do_sample=False
)

for i in range(len(sample_reviews)):
    print("ORIGINAL REVIEW:\n", sample_reviews[i])
    print("\nSUMMARY:\n", summaries[i]['summary_text'])
    print("=" * 80)


Your max_length is set to 60, but your input_length is only 3. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=1)
Your max_length is set to 60, but your input_length is only 4. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length is set to 60, but your input_length is only 7. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)
Your max_length is set to 60, but your input_length is only 5. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=2)
Your max_length 

ORIGINAL REVIEW:
 brilliant

SUMMARY:
 CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Please submit your best shots of London for next week. Visit CNN.com/Travel next Wednesday for a new gallery of snapshots.
ORIGINAL REVIEW:
 highly recommended

SUMMARY:
 CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Please submit your best shots for next week. Visit CNN.com/Travel next Wednesday for a new gallery of snapshots.
ORIGINAL REVIEW:
 value-for-money

SUMMARY:
 CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Visit CNN.com/Travel each week for a new gallery of snapshots.
ORIGINAL REVIEW:
 perfect product!

SUMMARY:
 The perfect product for the perfect person. The perfect way to make the perfect cup of tea. The. perfect product. For more information on this product, go to www.theperfectcup.com.
ORIGINAL REVIEW:
 wonderful

SUMMARY:
 Wife and mother-of-three has been described as "wonderful" and "lovely" by 

# Review Length Reduction

In [4]:
original_lengths = [len(r) for r in sample_reviews]
summary_lengths = [len(s['summary_text']) for s in summaries]

pd.DataFrame({
    "Original Length": original_lengths,
    "Summary Length": summary_lengths
})


,Original Length,Summary Length
0,9,198
1,18,188
2,15,138
3,16,180
4,9,140


# Save Example Comparisons

In [5]:
comparison_df = pd.DataFrame({
    "original_review": sample_reviews,
    "generated_summary": [s['summary_text'] for s in summaries]
})

comparison_df.to_csv("../results/summaries/summarization_experiments.csv", index=False)
